In [1]:
import pandas as pd
import json
from datetime import datetime
import ipaddress

# --- Configuration ---
SYNTHETIC_DATA_FILE = '/home/student/cyberai-lab/generated_data/synthetic_firewall_logs.jsonl'

print(f"If you see this message → your lab is working perfectly!")

try:
    # Load the synthetic data
    print(f"Loading synthetic data from {SYNTHETIC_DATA_FILE}...")
    data = []
    with open(SYNTHETIC_DATA_FILE, 'r') as f:
        for line in f:
            data.append(json.loads(line))

    df = pd.DataFrame(data)
    print(f"Successfully loaded {len(df)} log entries.")

    # --- Basic Format and Structure Validation ---
    print("\n--- Performing Basic Validation ---")

    # Check if DataFrame is empty
    if df.empty:
        print("Error: No data loaded. Cannot perform validation.")
    else:
        # Check for required columns
        required_columns = ["timestamp", "source_ip", "destination_ip", "destination_port", "protocol", "action", "reason"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Warning: Missing required columns: {missing_columns}")
        else:
            print("All required columns are present.")

        # Check data types and validity
        # Timestamp validation
        try:
            df['timestamp_dt'] = pd.to_datetime(df['timestamp'], errors='coerce')
            invalid_timestamps = df[df['timestamp_dt'].isna()]['timestamp'].tolist()
            if invalid_timestamps:
                print(f"Warning: Found {len(invalid_timestamps)} invalid timestamp formats. Examples: {invalid_timestamps[:3]}")
            else:
                print("Timestamp formats appear valid.")
        except Exception as e:
            print(f"Error during timestamp validation: {e}")

        # IP Address validation
        valid_ips = []
        invalid_ips = []
        for ip_col in ['source_ip', 'destination_ip']:
            for ip in df[ip_col]:
                try:
                    ipaddress.ip_address(ip)
                    valid_ips.append(ip)
                except ValueError:
                    invalid_ips.append(ip)
            if invalid_ips:
                print(f"Warning: Found {len(invalid_ips)} invalid IP addresses in '{ip_col}'. Examples: {invalid_ips[:3]}")
                invalid_ips.clear() # Reset for next column
            else:
                print(f"IP addresses in '{ip_col}' appear valid.")

        # Port validation (should be integer)
        try:
            df['destination_port_int'] = pd.to_numeric(df['destination_port'], errors='coerce')
            invalid_ports = df[df['destination_port_int'].isna()]['destination_port'].tolist()
            if invalid_ports:
                print(f"Warning: Found {len(invalid_ports)} invalid destination ports. Examples: {invalid_ports[:3]}")
            else:
                print("Destination ports appear valid.")
        except Exception as e:
            print(f"Error during port validation: {e}")

        # Action validation
        valid_actions = df['action'].unique()
        print(f"Unique actions found: {valid_actions}")
        if 'DENY' not in valid_actions:
             print("Warning: 'DENY' action not found in generated logs, though expected.")

    # --- Content Realism Check (Sample Inspection) ---
    print("\n--- Sample Inspection (First 3 Entries) ---")
    if not df.empty:
        print(df.head(3).to_markdown(index=False))
        print("\nReview the sample entries above for realism: Do IPs, ports, reasons, and timestamps look plausible?")
    else:
        print("No data to display sample.")

    # --- Statistical Analysis (Example: Action Distribution) ---
    print("\n--- Statistical Analysis ---")
    if not df.empty and 'action' in df.columns:
        action_counts = df['action'].value_counts()
        print("Distribution of actions:")
        print(action_counts)
        # Expected: Should show 'DENY' as the dominant or only action.
    else:
        print("Cannot perform action distribution analysis.")

    print("\nExpected output: Validation checks should pass with minimal warnings. Sample entries should look like realistic firewall logs. Action distribution should reflect the prompt (primarily 'DENY').")

except FileNotFoundError:
    print(f"Error: The file {SYNTHETIC_DATA_FILE} was not found. Please ensure the previous generation step completed successfully.")
except Exception as e:
    print(f"An unexpected error occurred during evaluation: {e}")

print("\n--- Lab Reset Command ---")
print("To reset the lab environment, run: docker compose down && docker compose up -d")

If you see this message → your lab is working perfectly!
Loading synthetic data from /home/student/cyberai-lab/generated_data/synthetic_firewall_logs.jsonl...
Error: The file /home/student/cyberai-lab/generated_data/synthetic_firewall_logs.jsonl was not found. Please ensure the previous generation step completed successfully.

--- Lab Reset Command ---
To reset the lab environment, run: docker compose down && docker compose up -d
